# End-to-End Clinical Triage Pipeline
**Stage 1 (Binary Gate):** Benign vs Malignant — 4-model stacking ensemble  
**Stage 2 (Severity Staging):** CORAL multiclass (AKIEC/BCC/SCC/MEL) — 3-model ensemble  
**XAI Layer:** Dual Grad-CAM++ heatmaps + MC-Dropout uncertainty for both stages  
**Output:** Per-patient clinical report PNG + system-wide evaluation CSV

In [1]:
# ── CELL 1 ─ Installs & Imports ─────────────────────────────────────────
!pip install -q timm albumentations scipy scikit-image

import os, gc, json, warnings, textwrap
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast
import torchvision.transforms as T
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import joblib

from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    classification_report, cohen_kappa_score
)

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'✅ Device: {DEVICE}')


✅ Device: cuda


In [2]:
# ── CELL 2 ─ Global Configuration ───────────────────────────────────────
# ── Kaggle paths — edit only these two sections ──────────────────────────
BASE_NPY    = '/kaggle/input/notebooks/nandini143'   # binary base model outputs
BINARY_STACKER = '/kaggle/input/notebooks/nandini143/stacking-ensemble/lr_stacker.pkl'

MULTI_CKPTS = [
    {'ckpt': '/kaggle/input/notebooks/nandini143/efficientnetv2-ordinal-severity/effnet_severity/effnet_severity_best.pth',   'weight': 0.7875},
    {'ckpt': '/kaggle/input/notebooks/nandini143/densenet-121-ordinal-severity/densenet_severity/densenet_severity_best.pth', 'weight': 0.7787},
    {'ckpt': '/kaggle/input/notebooks/nandini143/swin-tiny-ordinal-severity/swin_severity/swin_severity_best.pth',           'weight': 0.7993},
]

BINARY_NPY_FILES = {
    'densenet121'    : ('binary-densenet-121/densenet121_val_probs.npy',
                        'binary-densenet-121/densenet121_val_labels.npy',
                        'binary-densenet-121/densenet121_test_probs.npy'),
    'convnext'       : ('binary-convnext/convnext_val_probs.npy',
                        'binary-convnext/convnext_val_labels.npy',
                        'binary-convnext/convnext_test_probs.npy'),
    'efficientnetv2s': ('binary-efficientnetv2s/efficientnetv2s_val_probs.npy',
                        'binary-efficientnetv2s/efficientnetv2s_val_labels.npy',
                        'binary-efficientnetv2s/efficientnetv2s_test_probs.npy'),
    'swintiny'       : ('binary-swin-tiny-transformer/swintiny_val_probs.npy',
                        'binary-swin-tiny-transformer/swintiny_val_labels.npy',
                        'binary-swin-tiny-transformer/swintiny_test_probs.npy'),
}

# ── Data paths ────────────────────────────────────────────────────────────
MULTI_CSV   = '/kaggle/input/datasets/nandini143/multiclass-output/severity_data/severity_train_folds.csv'
MULTI_IMGDIR= '/kaggle/input/datasets/nandini143/multiclass-output/severity_data/preprocessed_images'
BINARY_CSV  = '/kaggle/input/notebooks/nandini143/data-preparation/binary_test.csv'
OUTPUT_DIR  = '/kaggle/working/clinical_triage'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ── Thresholds ────────────────────────────────────────────────────────────
BINARY_THRESHOLD    = 0.50          # P(Malignant) >= this → triggers staging
CORAL_THRESHOLDS    = [0.460, 0.506, 0.501]  # t1, t2, t3 (Strategy 1 optimal)
UNCERTAINTY_THRESH  = 0.02          # MC-Dropout variance above this → flag
AUTO_DIAG_CONF      = 0.80          # must exceed this for Auto-Diagnosis

# ── Inference settings ────────────────────────────────────────────────────
IMG_SIZE_CNN   = 384
IMG_SIZE_SWIN  = 224
BATCH_SIZE     = 16
MC_RUNS        = 30
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]
STAGE_NAMES    = {0: 'AKIEC', 1: 'BCC', 2: 'SCC', 3: 'MEL'}

print('✅ Config ready.')


✅ Config ready.


In [3]:
# ── CELL 3 ─ Binary Model Architectures ─────────────────────────────────
# Exact architectural mirrors of the base training notebooks

class SwinPooler(nn.Module):
    def forward(self, x):
        if len(x.shape) == 4: return x.mean(dim=(1, 2))
        return x.mean(dim=1)

def build_densenet121():
    m = timm.create_model('densenet121', pretrained=False, num_classes=0, global_pool='avg')
    in_f = m.num_features
    m.classifier = nn.Sequential(
        nn.BatchNorm1d(in_f), nn.Dropout(0.4),
        nn.Linear(in_f, 256), nn.ReLU(),
        nn.Dropout(0.3), nn.Linear(256, 1)
    )
    return m

def build_efficientnetv2s():
    m = timm.create_model('tf_efficientnetv2_s', pretrained=False,
                          num_classes=0, global_pool='avg', drop_path_rate=0.2)
    in_f = m.num_features
    m.classifier = nn.Sequential(
        nn.BatchNorm1d(in_f), nn.Dropout(0.4),
        nn.Linear(in_f, 256), nn.ReLU(),
        nn.Dropout(0.3), nn.Linear(256, 1)
    )
    return m

def build_convnext():
    m = timm.create_model('convnext_tiny', pretrained=False, num_classes=0, global_pool='avg')
    in_f = m.num_features
    m.head.fc = nn.Sequential(
        nn.BatchNorm1d(in_f), nn.Dropout(0.4),
        nn.Linear(in_f, 256), nn.ReLU(),
        nn.Dropout(0.3), nn.Linear(256, 1)
    )
    return m

def build_swin_tiny():
    m = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False,
                          num_classes=0, drop_path_rate=0.2)
    in_f = m.num_features
    m.head = nn.Sequential(
        SwinPooler(),
        nn.LayerNorm(in_f), nn.Dropout(0.4),
        nn.Linear(in_f, 256), nn.GELU(),
        nn.Dropout(0.3), nn.Linear(256, 1)
    )
    return m

def load_binary_model(builder, ckpt_path, device):
    model = builder()
    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state)
    return model.to(device).eval()

print('✅ Binary architectures defined.')


✅ Binary architectures defined.


In [4]:
# ── CELL 4 ─ Multiclass CORAL Architecture ───────────────────────────────

class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, out_dim=32, dropout=0.2, use_batchnorm=True):
        super().__init__()
        act  = nn.ReLU(inplace=True) if use_batchnorm else nn.GELU()
        norm = nn.BatchNorm1d(hidden_dim) if use_batchnorm else nn.LayerNorm(hidden_dim)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), norm, act,
            nn.Dropout(dropout), nn.Linear(hidden_dim, out_dim), act
        )
    def forward(self, x): return self.net(x)

class SeverityModel(nn.Module):
    def __init__(self, model_name, num_outputs, meta_dim, head_type='layernorm', pretrained=False):
        super().__init__()
        self.backbone  = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        feat_dim       = self.backbone.num_features
        self.use_meta  = meta_dim > 0
        if self.use_meta:
            self.meta_mlp = TabularMLP(meta_dim, 64, 32, 0.2, True)
            fusion_dim = feat_dim + 32
        else:
            fusion_dim = feat_dim
        if head_type == 'batchnorm':
            self.head = nn.Sequential(
                nn.BatchNorm1d(fusion_dim),
                nn.Linear(fusion_dim, 512), nn.ReLU(inplace=True), nn.Dropout(0.3),
                nn.Linear(512, 256), nn.ReLU(inplace=True), nn.Dropout(0.15),
                nn.Linear(256, num_outputs)
            )
        elif head_type == 'swin':
            self.head = nn.Sequential(
                nn.LayerNorm(fusion_dim),
                nn.Linear(fusion_dim, 512), nn.GELU(), nn.Dropout(0.3),
                nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.1),
                nn.Linear(128, num_outputs)
            )
        else:
            self.head = nn.Sequential(
                nn.LayerNorm(fusion_dim),
                nn.Linear(fusion_dim, 256), nn.GELU(), nn.Dropout(0.3),
                nn.Linear(256, num_outputs)
            )
    def forward(self, images, meta):
        feat = self.backbone(images)
        if self.use_meta:
            feat = torch.cat([feat, self.meta_mlp(meta)], dim=1)
        return self.head(feat)

def load_severity_model(ckpt_path):
    ckpt        = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model_name  = ckpt['model_name']
    meta_dim    = ckpt.get('meta_dim', 0)
    num_outputs = ckpt.get('num_outputs', 3)
    site_cols   = ckpt.get('site_cols', [])
    if 'densenet' in model_name.lower():  head_type = 'batchnorm'
    elif 'swin'   in model_name.lower():  head_type = 'swin'
    else:                                  head_type = 'layernorm'
    model = SeverityModel(model_name, num_outputs, meta_dim, head_type, False).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    qwk = ckpt.get('qwk', 'N/A')
    print(f'  ✅ {model_name} | meta_dim={meta_dim} | head={head_type} | QWK={qwk:.4f}' if isinstance(qwk, float) else f'  ✅ {model_name} | meta_dim={meta_dim} | head={head_type}')
    return model, site_cols, num_outputs, meta_dim

def coral_to_label(logits, thresholds):
    sigs = torch.sigmoid(logits).cpu().numpy()
    t = np.array(thresholds)
    return (sigs > t).sum(axis=1).astype(int)

print('✅ Multiclass CORAL architecture defined.')


✅ Multiclass CORAL architecture defined.


In [5]:
# ── CELL 5 ─ XAI Engine: Dual Grad-CAM++ & Score-CAM ───────────────────

class GradCAMPP:
    """SmoothGrad-CAM++ for CNN backbones (DenseNet, EfficientNet, ConvNeXt)."""
    def __init__(self, model, target_layer, n_smooth=5, noise_sigma=0.02):
        self.model, self.n_smooth, self.noise_sigma = model, n_smooth, noise_sigma
        self.activations = self.gradients = None
        self._fh = target_layer.register_forward_hook(self._save_act)
        self._bh = target_layer.register_full_backward_hook(self._save_grad)

    def _reshape(self, x):
        if len(x.shape) == 3:  # (B, L, C) → (B, C, H, W)
            B, L, C = x.shape; H = W = int(L**0.5)
            return x.transpose(1, 2).reshape(B, C, H, W)
        return x

    def _save_act(self, m, i, o):  self.activations = self._reshape(o.detach())
    def _save_grad(self, m, i, o): self.gradients   = self._reshape(o[0].detach())

    def generate(self, img_t):
        cams = []
        for _ in range(self.n_smooth):
            noisy = (img_t + torch.randn_like(img_t) * self.noise_sigma).detach().requires_grad_(True)
            self.model.zero_grad()
            out = self.model(noisy)
            if isinstance(out, tuple): out = out[0]
            out.squeeze().sum().backward()
            a, g = self.activations, self.gradients
            if a is None or g is None: continue
            gsq   = g**2
            alpha = gsq / (2*gsq + (a * g**3).sum(dim=(2,3), keepdim=True) + 1e-8)
            w     = (alpha * F.relu(g)).sum(dim=(2,3), keepdim=True)
            cam   = F.relu((w * a).sum(dim=1)).squeeze(0).cpu().numpy()
            if cam.max() > 0: cam = (cam - cam.min()) / (cam.max() - cam.min())
            cams.append(cam)
        if not cams: return np.zeros((7, 7))
        avg = np.mean(cams, axis=0)
        if avg.max() > 0: avg = (avg - avg.min()) / (avg.max() - avg.min())
        return avg

    def remove(self): self._fh.remove(); self._bh.remove()


class SwinAttentionMap:
    """Attention Rollout for Swin Transformer (no gradient needed)."""
    @staticmethod
    def generate(model, img_t):
        attn_list, hooks = [], []
        def hook(m, inp, out):
            if isinstance(inp, tuple) and len(inp) >= 1:
                x = inp[0]; B, N, C = x.shape
                xn = F.normalize(x, dim=-1)
                attn = F.softmax(torch.bmm(xn, xn.transpose(1,2)) / (C**0.5), dim=-1)
                attn_list.append(attn.detach().cpu())
        for nm, m in model.named_modules():
            if 'attn' in nm.lower() and hasattr(m, 'num_heads'):
                hooks.append(m.register_forward_hook(hook))
        with torch.no_grad(): model(img_t)
        for h in hooks: h.remove()
        if not attn_list: return np.ones((7, 7)) / 49
        rollout = None
        for a in attn_list:
            am = a.mean(0); am = am + torch.eye(am.shape[0])
            am = am / am.sum(-1, keepdim=True)
            rollout = am if rollout is None else torch.mm(rollout, am)
        patch = rollout[0, 1:] if rollout.shape[0] > 1 else rollout[0]
        s = int(patch.shape[0]**0.5)
        cam = patch[:s*s].numpy().reshape(s, s)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def get_target_layer(model_name_str, model):
    """Auto-detect the best Grad-CAM target layer."""
    mn = model_name_str.lower()
    if 'densenet'    in mn: return model.features.denseblock4
    if 'efficientnet' in mn: return model.conv_head
    if 'convnext'    in mn:
        stages = list(model.stages.children())
        return stages[-1]
    return None  # Swin → use attention rollout


def upsample_heatmap(cam, h, w):
    t = torch.from_numpy(cam.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    return F.interpolate(t, size=(h, w), mode='bilinear', align_corners=False).squeeze().numpy()


def fuse_heatmaps(hmap_list, target=(224, 224)):
    """Upsample and mean-fuse heatmaps from multiple models."""
    ups = []
    for h in hmap_list:
        u = upsample_heatmap(h, *target)
        if u.max() > 0: u = (u - u.min()) / (u.max() - u.min())
        ups.append(u)
    fused = np.mean(ups, axis=0)
    fused = (fused - fused.min()) / (fused.max() - fused.min() + 1e-8)
    return fused

print('✅ XAI engine ready (GradCAM++, Attention Rollout, Fusion).')


✅ XAI engine ready (GradCAM++, Attention Rollout, Fusion).
